In [1]:
import pymysql

conn = pymysql.connect(host='localhost', user='root', password='1111', database='finance',
                       charset='utf8')
cur = conn.cursor()

In [2]:
from decimal import Decimal, InvalidOperation
from datetime import datetime, date
from pathlib import Path
from dataclasses import dataclass
import csv
import re
import pandas as pd
from IPython.display import display

# 자료 파일 위치: 프로젝트 루트 또는 notebooks 폴더에서 실행 가능
_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'finance-sql']
ROOT = next((p for p in _candidates if (p / 'data/statement.csv').is_file()
             and (p / 'finance_helpers.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('전체 ZIP을 풀고 pymysql_finance_course 폴더에서 Jupyter를 시작하세요.')

In [3]:
cur.execute('SELECT account_id, currency, balance FROM fin_accounts WHERE account_id=%s', (101,))
account = cur.fetchone()
print(account)
assert account == (101, 'KRW', Decimal('1120000.00'))

(101, 'KRW', Decimal('1120000.00'))


* SQL 주입처럼 보이는 문자열도 값으로 조회

In [4]:
suspicious_value = "E001' OR 1=1 -- "
cur.execute('SELECT external_id, amount FROM fin_ledger_entries WHERE external_id=%s',
            (suspicious_value,))
safe_rows = cur.fetchall()
print('주입 문자열 안전 조회:', safe_rows)
assert not safe_rows
cur.execute('SELECT external_id, amount FROM fin_ledger_entries WHERE external_id=%s', ('E001',))
assert cur.fetchone() == ('E001', Decimal('200000.00'))

주입 문자열 안전 조회: ()


* 고객 → 계좌 → 원장 JOIN


In [5]:
join_sql = """
SELECT c.customer_alias, a.account_id, l.external_id, l.posted_at, l.amount
FROM fin_customers AS c
JOIN fin_accounts AS a ON a.customer_id = c.customer_id
JOIN fin_ledger_entries AS l ON l.account_id = a.account_id
WHERE l.status = %s
ORDER BY l.posted_at, l.entry_id
"""
cur.execute(join_sql, ('posted',))
join_columns = [item[0] for item in cur.description]
joined_rows = cur.fetchall()
print('열:', join_columns)
for row in joined_rows:
    print(row)
assert len(joined_rows) == 7
assert {row[0] for row in joined_rows} == {'고객A', '고객B', '고객C'}

열: ['customer_alias', 'account_id', 'external_id', 'posted_at', 'amount']
('고객A', 101, 'E001', datetime.datetime(2025, 1, 10, 9, 0), Decimal('200000.00'))
('고객A', 101, 'E002', datetime.datetime(2025, 1, 10, 12, 0), Decimal('-50000.00'))
('고객A', 101, 'E003', datetime.datetime(2025, 1, 11, 10, 0), Decimal('-30000.00'))
('고객B', 102, 'E004', datetime.datetime(2025, 1, 11, 10, 0), Decimal('30000.00'))
('고객B', 102, 'E005', datetime.datetime(2025, 1, 11, 14, 0), Decimal('100000.00'))
('고객B', 102, 'E006', datetime.datetime(2025, 1, 12, 11, 0), Decimal('-40000.00'))
('고객C', 103, 'E007', datetime.datetime(2025, 1, 12, 13, 0), Decimal('-10000.00'))


* 튜플 결과를 DataFrame으로 옮기는 실제 함수


In [6]:
def query_frame(cur, sql, params=None):
    cur.execute(sql, params)
    columns = [col[0] for col in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=columns)

* posted 전체 유입·유출·순변동


In [7]:
total_sql = """
SELECT COUNT(*) AS posted_count,
       COALESCE(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 0) AS inflow,
       COALESCE(SUM(CASE WHEN amount < 0 THEN -amount ELSE 0 END), 0) AS outflow,
       COALESCE(SUM(amount), 0) AS net_flow
FROM fin_ledger_entries WHERE status = %s
"""
cur.execute(total_sql, ('posted',))
totals = cur.fetchone()
print('건수·유입·유출·순변동:', totals)
assert totals == (7, Decimal('330000.00'), Decimal('130000.00'), Decimal('200000.00'))
assert totals[1] - totals[2] == totals[3]

건수·유입·유출·순변동: (7, Decimal('330000.00'), Decimal('130000.00'), Decimal('200000.00'))


* 일별 집계와 반열린 날짜 구간


In [8]:
daily_sql = """
SELECT DATE(posted_at) AS business_date, COUNT(*) AS posted_count,
       SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END) AS inflow,
       SUM(CASE WHEN amount < 0 THEN -amount ELSE 0 END) AS outflow,
       SUM(amount) AS net_flow
FROM fin_ledger_entries
WHERE status = %s AND posted_at >= %s AND posted_at < %s
GROUP BY DATE(posted_at)
ORDER BY business_date
"""
daily = query_frame(cur, daily_sql, ('posted', '2025-01-10', '2025-01-13'))
display(daily)
assert daily['posted_count'].tolist() == [2, 3, 2]
assert daily['net_flow'].tolist() == [Decimal('150000.00'), Decimal('100000.00'), Decimal('-50000.00')]
assert all(isinstance(x, Decimal) for x in daily['net_flow'])
assert sum(daily['net_flow'], Decimal('0.00')) == totals[3]

,business_date,posted_count,inflow,outflow,net_flow
0,2025-01-10,2,200000.00,50000.00,150000.00
1,2025-01-11,3,130000.00,30000.00,100000.00
2,2025-01-12,2,0.00,50000.00,-50000.00


* 기초잔액 검산 SQL과 함수 본문



In [9]:
BALANCE_SQL = """
SELECT a.account_id, a.currency, a.opening_balance, a.balance,
       COALESCE(SUM(l.amount), 0) AS posted_net,
       a.opening_balance + COALESCE(SUM(l.amount), 0) AS expected_balance
FROM fin_accounts AS a
LEFT JOIN fin_ledger_entries AS l ON l.account_id=a.account_id AND l.status=%s
GROUP BY a.account_id, a.currency, a.opening_balance, a.balance
ORDER BY a.account_id
"""

def check_balances(cur):
    checks = query_frame(cur, BALANCE_SQL, ('posted',))
    checks['difference'] = checks['balance'] - checks['expected_balance']
    return checks

* 검산 실행과 기대값


In [10]:
checks = check_balances(cur)
display(checks)
assert checks['account_id'].tolist() == [101, 102, 103]
assert checks['posted_net'].tolist() == [Decimal('120000.00'), Decimal('90000.00'), Decimal('-10000.00')]
assert all(value == Decimal('0.00') for value in checks['difference'])
assert sum(checks['balance'], Decimal('0.00')) == (
    sum(checks['opening_balance'], Decimal('0.00')) + totals[3])

,account_id,currency,opening_balance,balance,posted_net,expected_balance,difference
0,101,KRW,1000000.00,1120000.00,120000.00,1120000.00,0.00
1,102,KRW,500000.00,590000.00,90000.00,590000.00,0.00
2,103,KRW,200000.00,190000.00,-10000.00,190000.00,0.00


* 상태를 혼동하지 않기



In [11]:
status_summary = query_frame(cur,
    'SELECT status, COUNT(*) AS entry_count, SUM(amount) AS net_flow '
    'FROM fin_ledger_entries GROUP BY status ORDER BY status')
display(status_summary)
cur.execute('SELECT SUM(amount) FROM fin_ledger_entries WHERE transfer_request_id=%s AND status=%s',
            ('seed-t01', 'posted'))
assert cur.fetchone()[0] == Decimal('0.00')
assert conn.get_autocommit() is False

,status,entry_count,net_flow
0,pending,1,50000.00
1,posted,7,200000.00
2,reversed,1,-20000.00


In [12]:
conn.rollback()  # 남아 있는 미확정 변경/읽기 트랜잭션 종료
cur.close()
conn.close()
print('cur와 conn을 닫았습니다. 다시 실행하려면 첫 연결 셀부터 실행하세요.')

cur와 conn을 닫았습니다. 다시 실행하려면 첫 연결 셀부터 실행하세요.
